# Notebook 11 — Scaling Laws and Training-Compute Planning

    ## Learning objectives

    - Estimate parameter, token, memory, and FLOP budgets
- Fit cautious empirical scaling curves
- Design pilot runs and stopping decisions

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

# VS Code's Colab extension can attach to a Colab kernel before `google.colab`
# has been imported, so checking only sys.modules produces a false negative.
try:
    HAS_GOOGLE_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # The parent `google` namespace is absent locally.
    HAS_GOOGLE_COLAB = False
IN_COLAB = HAS_GOOGLE_COLAB or bool(os.getenv("COLAB_RELEASE_TAG")) or bool(os.getenv("COLAB_GPU"))
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if IN_COLAB and not token:
    from google.colab import userdata
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            token = None
        if token:
            break
elif not IN_COLAB:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")

# `HF_TOKEN` is the canonical huggingface_hub variable. The course also sets its
# descriptive alias because some lesson code uses HUGGINGFACE_TOKEN explicitly.
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGINGFACE_TOKEN"] = token

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Colab runtime detected:", IN_COLAB)
print("Hugging Face token configured:", bool(os.getenv("HF_TOKEN")))
if IN_COLAB and not token:
    print("Add an HF_TOKEN secret in Colab, enable notebook access, then rerun this cell.")


## 11.1 What scales

Model size, active parameters, training tokens, sequence length, batch tokens, optimizer updates, numerical precision, and hardware utilization are different quantities. Dense decoder training FLOPs are often estimated with a constant times parameters times tokens, but embeddings, attention, MoE routing, rematerialization, and inefficient kernels change the constant. Report assumptions and measured throughput. Epochs are misleading across corpora; tokens and compute make experiments comparable. Storage also includes raw/tokenized data, optimizer state, checkpoints, logs, and evaluation artifacts.


In [ ]:
def dense_flops(params,tokens,multiplier=6): return multiplier*params*tokens
for p,t in [(1e8,2e9),(1e9,2e10),(7e9,1.4e11)]: print(p,t,dense_flops(p,t))


## 11.2 Empirical power laws

Loss frequently follows approximate power laws over useful scale ranges, with an irreducible floor. Fit in log space only after plotting residuals and separating architecture or data-regime changes. A smooth aggregate curve can hide language, domain, memorization, safety, or downstream regressions. Extrapolation beyond observed compute has wide structural uncertainty. Scaling laws help allocate pilots and detect underperforming runs; they do not replace evaluation or guarantee emergent behavior.


In [ ]:
import numpy as np
compute=np.array([1,2,4,8,16.]); loss=2.1+.8*compute**-.3
coef=np.polyfit(np.log(compute),np.log(loss-2.1),1); print("exponent",coef[0])


## 11.3 Compute-optimal tradeoffs

Given fixed compute, increasing parameters leaves fewer tokens and increasing tokens leaves a smaller model. Compute-optimal guidance depends on data quality, reuse, inference demand, and target capabilities. A smaller model trained longer can be preferable when serving dominates lifetime cost; a larger undertrained model may adapt differently. Deduplicated unique tokens, repeated tokens, synthetic mixtures, and curriculum order are not interchangeable. Treat token-to-parameter ratios as hypotheses to test with pilots, not universal constants.


In [ ]:
budget=6e20
for params in [1e8,5e8,1e9,3e9]: print(params,"tokens",budget/(6*params))


## 11.4 Planning and monitoring

Run small matched pilots varying one scale axis, fit curves with uncertainty, and reserve budget for failures, evaluation, checkpointing, and ablations. Estimate memory for weights, gradients, optimizer states, activations, KV/cache workspaces, and fragmentation. During training compare observed loss with predicted bands, tokens per second, utilization, gradient norms, and data health. Stop or intervene on evidence, not sunk cost. Archive failed runs and document carbon, financial, and opportunity costs alongside quality.


In [ ]:
def gib(params,weight=2,grad=2,optimizer=8): return params*(weight+grad+optimizer)/2**30
for p in [1e8,1e9,7e9]: print(p,gib(p))


## Reference workflow and evidence standard

Treat the notebook as an experiment, not a recipe. State the question, freeze inputs and
success criteria, establish the simplest baseline, change one material factor, and retain raw
outputs needed to diagnose failures. Record model, tokenizer, template, data and code revisions;
hardware and dtype; random seeds; generation or optimization configuration; token counts;
latency and memory; and results by meaningful slice. A demonstration that runs is evidence of
plumbing, not evidence of general capability.

Test boundaries as well as the happy path: empty and maximum-length inputs, malformed records,
multilingual or code text, unavailable dependencies, cancellation, and adversarial content.
Keep credentials in environment or Colab Secrets and never serialize them with artifacts. Pin
remote revisions, review licenses and custom code, validate saved artifacts in a fresh process,
and prefer deterministic validators wherever outputs can be checked mechanically.

Before applying the technique, compare it with prompting, retrieval, a smaller model, or no
model. Report quality together with compute, storage, latency, and operational complexity. Use
held-out data and paired comparisons, disclose uncertainty and negative results, and define a
rollback path. These practices connect low-level understanding to reliable application work.

A useful completion checklist asks four separate questions. Is the mathematical contract clear
enough to predict shapes, masks, reductions, and failure cases? Does the implementation reproduce
a tiny hand-worked or deterministic reference? Does the measured result survive a held-out set,
relevant slices, and an ablation against a simpler baseline? Can another person reload the exact
artifacts and reconstruct the claim from the manifest? Passing only the first two establishes a
tutorial demonstration; passing all four supports an engineering decision. When a result fails,
preserve the counterexample and update the test suite before changing the implementation.

Finally, separate correctness, capability, efficiency, and safety conclusions. A correct
implementation may have weak capability; a capable prototype may be too costly or unsafe to
deploy. Name the population to which each conclusion applies and avoid converting a single
metric into a universal ranking. Track assumptions beside results, especially tokenizer and
template compatibility, data rights, access-control boundaries, and hardware-specific behavior.
Leave exercises with an executable acceptance criterion, a baseline result, and a short written
interpretation. That combination turns exploratory code into cumulative course evidence that can
be revisited when libraries, model families, or deployment engines change.


## 11.5 Fit pilots with uncertainty

A scaling fit should preserve uncertainty from finite evaluation data, random seeds, optimization noise, and model misspecification. Fit multiple candidate forms, inspect residuals, and use bootstrap resampling of runs or evaluation items. Avoid estimating both a loss floor and exponent from too few scales: the parameters become highly correlated. Predictions outside the measured range are scenarios, not facts. Use intervals to decide whether a larger run is distinguishable from a cheaper alternative and predefine stop or escalation criteria. Store every pilot, including failed runs, because excluding them creates an unrealistically smooth frontier.


In [ ]:
rng=np.random.default_rng(7); compute=np.array([1.,2.,4.,8.,16.]); observed=2.0+.9*compute**-.28+rng.normal(0,.015,len(compute))
slopes=[]
for _ in range(500):
 idx=rng.integers(0,len(compute),len(compute)); x=np.log(compute[idx]); y=np.log(np.maximum(observed[idx]-2.0,1e-6))
 if np.std(x)>0: slopes.append(np.polyfit(x,y,1)[0])
print("slope median/interval",np.quantile(slopes,[.5,.05,.95]))


## 11.6 End-to-end budget worksheet

Training compute is only one part of an experiment budget. Include tokenizer and data processing, ablations, evaluation generation, checkpoints, failed runs, hyperparameter pilots, and engineer time. For deployment-aware planning, estimate lifetime inference tokens, prefill/decode mix, replicas, utilization, and latency constraints. A somewhat more expensive training run can be economical if it creates a smaller capable model; conversely, distillation costs may not break even at low request volume. State monetary and energy assumptions as ranges rather than false precision, and update the worksheet with measured throughput after the first pilot.


In [ ]:
budget={"main_train":6000,"pilots":1800,"data":500,"evaluation":700,"failures_reserve":1500,"storage":300}
total=sum(budget.values()); print("planned",total,budget)
actual_tokens_per_second=18000; train_tokens=2_000_000_000; accelerator_hours=train_tokens/actual_tokens_per_second/3600
print("accelerator-hours at measured throughput",accelerator_hours)


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361)
- [Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556)


## Exercises

    1. Fit a power law and bootstrap its exponent.
2. Plan three matched pilot runs.
3. Compare training and lifetime inference compute.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
